<a id="encrypted-machine-learning-models"></a>
# Encrypted Machine Learning: Models

This tutorial covers `src/concrete_fhe_toolkit/ml/models.py`. This massive module provides the raw algorithmic inference functions for all supported ML models. These functions are what the `FHEModel` classes compile under the hood.

<a id="knn-inference"></a>
## KNN Inference

KNN inference works by calculating distances between the encrypted test sample and all public training samples, and returning the label of the nearest neighbor (or majority vote if k > 1).

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.models import knn_inference

def test_knn(feat1: int, feat2: int):
    # Public training data
    train_samples = [[0, 0], [10, 10]]
    train_labels = [0, 1]
    
    return knn_inference(
        test_sample=[feat1, feat2], 
        train_samples=train_samples, 
        train_labels=train_labels, 
        k=1, 
        max_distance=300
    )

compiler = fhe.Compiler(test_knn, {"feat1": "encrypted", "feat2": "encrypted"})
circuit = compiler.compile([(1, 1), (9, 9)])

assert circuit.encrypt_run_decrypt(1, 1) == 0  # Near (0,0)
assert circuit.encrypt_run_decrypt(9, 9) == 1  # Near (10,10)
print("✅ Encrypted KNN Inference passed!")

<a id="decision-tree-inference"></a>
## Decision Tree Inference

Decision Trees are executed obliviously. Every path is evaluated, and the final result is selected without leaking which branch was taken.

In [ ]:
from concrete_fhe_toolkit.ml.models import decision_tree_inference

def test_tree(feat: int):
    tree_def = {
        "feature": 0, "threshold": 5, 
        "left": 100,  # Value if feat >= 5
        "right": 200  # Value if feat < 5
    }
    return decision_tree_inference([feat], tree_def)

compiler = fhe.Compiler(test_tree, {"feat": "encrypted"})
circuit = compiler.compile([(1,), (10,)])

assert circuit.encrypt_run_decrypt(7) == 100
assert circuit.encrypt_run_decrypt(2) == 200
print("✅ Encrypted Decision Tree Inference passed!")

<a id="naive-bayes-inference"></a>
## Naive Bayes Inference

Evaluates a Naive Bayes model using quantized log probabilities.

In [ ]:
from concrete_fhe_toolkit.ml.models import naive_bayes_inference

def test_nb(feat: int):
    # 2 Classes. 1 Feature (binary: 0 or 1)
    # Class 0: High prob for feat=0 (log prob -1), Low prob for feat=1 (log prob -10)
    # Class 1: Low prob for feat=0 (log prob -10), High prob for feat=1 (log prob -1)
    log_prob_tables = [
        [[-1, -10]], # Class 0 tables
        [[-10, -1]]  # Class 1 tables
    ]
    priors = [0, 0] # Equal priors
    
    return naive_bayes_inference([feat], log_prob_tables, priors, min_feature=0)

compiler = fhe.Compiler(test_nb, {"feat": "encrypted"})
circuit = compiler.compile([(0,), (1,)])

assert circuit.encrypt_run_decrypt(0) == 0
assert circuit.encrypt_run_decrypt(1) == 1
print("✅ Encrypted Naive Bayes Inference passed!")